# Penyeimbangan Data Ecoli (ADASYN)

# VISUALISASI DISTRIBUSI KELAS


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from sqlalchemy import create_engine
from collections import Counter
from imblearn.over_sampling import ADASYN  # 🔹 ganti ke ADASYN

# === 1. Koneksi ke MySQL XAMPP ===
engine = create_engine("mysql+pymysql://root:@localhost:3306/ecoli")

# ambil data dari tabel
df = pd.read_sql("SELECT * FROM ecoli;", engine)
class_counts = df['class_label'].value_counts().sort_index()
nt = df.drop(columns=["class_label", "id_protein"])
ns = df["class_label"]

# ==== PCA data asli ====

pca = PCA(n_components=2)
X_pca = pca.fit_transform(nt)

plt.figure(figsize=(8,6))
for label in np.unique(ns):
    idx = (ns == label)
    plt.scatter(X_pca[idx,0], X_pca[idx,1], label=label, alpha=0.7)
plt.title("PCA Scatter Plot (Original Data)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.legend(); plt.show()

# ==== FIX DATA KOTOR ====
mask = ns != "class_label"
nt = nt[mask]
ns = ns[mask]

temp = sorted(class_counts)
temp.pop(0)

for i in range(0, 7):
    n = temp[i] - 1

    nt, ns = ADASYN(n_neighbors=n, sampling_strategy='minority').fit_resample(nt, ns)

df_resampled = pd.DataFrame(nt)  
df_resampled["class_label"] = ns  

df_resampled.to_csv("hasil_oversampling_ADASYN.csv", index=False)

print(f"{sorted(Counter(ns).items())}")

# === 2. PCA ===
pca = PCA(n_components=2)     
principalComponents = pca.fit_transform(nt)
principalDf = pd.DataFrame(data=principalComponents, columns=['principal component 1', 'principal component 2'])
finalDf = pd.concat([principalDf, ns.reset_index(drop=True)], axis=1)
print(finalDf)
# === 3. Visualisasi ===
fig, ax = plt.subplots(figsize=(10, 6))
targets = sorted(finalDf['class_label'].unique())
colors = plt.cm.get_cmap('tab10', len(targets))
for i, target in enumerate(targets):
    indicesToKeep = finalDf['class_label'] == target
    ax.scatter(finalDf.loc[indicesToKeep, 'principal component 1'], 
               finalDf.loc[indicesToKeep, 'principal component 2'], 
               c=[colors(i)], s=50, label=target)
ax.legend()
ax.grid()
plt.title('PCA of Ecoli Dataset after ADASYN Oversampling')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')

plt.show()



AttributeError: 'OptionEngine' object has no attribute 'execute'

# INFORMASI DATASET

In [5]:
print("\nInfo Dataset:")
df


Info Dataset:


,id_protein,feature1,feature2,feature3,feature4,feature5,feature6,feature7,class_label
0,AAS_ECOLI,0.44,0.52,0.48,0.5,0.43,0.47,0.54,im
1,AAT_ECOLI,0.49,0.29,0.48,0.5,0.56,0.24,0.35,cp
2,ACEA_ECOLI,0.07,0.40,0.48,0.5,0.54,0.35,0.44,cp
3,ACEK_ECOLI,0.56,0.40,0.48,0.5,0.49,0.37,0.46,cp
4,ACKA_ECOLI,0.59,0.49,0.48,0.5,0.52,0.45,0.36,cp
...,...,...,...,...,...,...,...,...,...
332,XYLA_ECOLI,0.16,0.43,0.48,0.5,0.54,0.27,0.37,cp
333,XYLE_ECOLI,0.69,0.39,0.48,0.5,0.57,0.76,0.79,im
334,XYLF_ECOLI,0.59,0.61,0.48,0.5,0.42,0.42,0.37,pp
335,YCEE_ECOLI,0.52,0.54,0.48,0.5,0.62,0.76,0.79,im
